# CS781 capstone - Phase 3: Rhetorical roles
Donkena Manideep (261040623).  Runs the code of `phase3/` from my submission on Kaggle.

**Before running:** turn on a GPU (T4 x2 or better; the code uses bfloat16), turn on Internet (for the weights and
`pip`), and attach as Kaggle datasets: (a) the code (the `code/` folder of my archive) unless the GitHub repository is
public, (b) the Phase 3 competition data.
Set `DATA_INPUT` below to the folder of (b).  `QUICK = True` runs short smoke tests; `False` runs the full recipe
(the full runs took 15-95 minutes each on an RTX PRO 5000 and take several times longer on a T4).

In [ ]:
PHASE = 3
QUICK = True                                    # False = the full recipe
DATA_INPUT = "/kaggle/input/cs781-phase3-data/Phase_3_RR"   # <- the attached competition data
DATA_SUBDIR = "Phase_3_RR"
REPO_URL = "https://github.com/ManideepDonkena/CS781-Legal-Intelligence-System"
REL = "https://github.com/ManideepDonkena/CS781-Legal-Intelligence-System/releases/download/models-v1/"

In [ ]:
# ---- setup: code, weights, folder layout ----------------------------------------------------------------------
import glob, os, shutil, subprocess, sys, urllib.request

def sh(cmd):
    """Run a shell command, print its output as it comes, stop on error."""
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")

# 1. the code: an attached Kaggle dataset that holds phase1..phase7 (the unzipped archive's code/ folder),
#    else a clone of the GitHub repository (the repository must be public for this)
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/phase1/final_model", recursive=True)]
if found:
    REPO = os.path.dirname(found[0])
else:
    REPO = "/kaggle/working/repo"
    if not os.path.isdir(REPO):
        sh(f"git clone --depth 1 {REPO_URL} {REPO}")
print("code:", REPO)
sh(f"pip install -q -r {REPO}/requirements.txt || true")

# 2. the layout the code expects
CAP = "/home/mani/cs781/Capstone"
LLM = "/home/mani/cs781_phase1/Legal_LLM"
os.makedirs(os.path.dirname(LLM), exist_ok=True)
if not os.path.exists(LLM):
    shutil.copytree(f"{REPO}/phase1/final_model", LLM)                 # model code (modern_arch.py, rope.py, ...)
NAMES = {2: "Phase2_LNER", 3: "Phase3_RR", 4: "Phase4_CJPE", 5: "Phase5_LSI", 6: "Phase6_SUM", 7: "Phase7_PCR"}
for p, name in NAMES.items():                                          # every phase's src (they import each other)
    os.makedirs(f"{CAP}/{name}/runs", exist_ok=True)
    if not os.path.exists(f"{CAP}/{name}/src"):
        shutil.copytree(f"{REPO}/phase{p}/src", f"{CAP}/{name}/src")
if not os.path.exists(f"{CAP}/shared"):
    shutil.copytree(f"{REPO}/shared", f"{CAP}/shared")

# 3. this phase's competition data (attach it as a Kaggle dataset; DATA_INPUT is its folder)
if PHASE > 1:
    target = f"{CAP}/{NAMES[PHASE]}/data/{DATA_SUBDIR}"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        os.symlink(DATA_INPUT, target)
    print("data:", target, "->", os.listdir(target)[:10])

# 4. the Phase 1 weights from the GitHub release models-v1 (about 1.2 GB)
W = "/workspace/models/cs781_phase1_runs/ROPEFIX_295M"
os.makedirs(W, exist_ok=True)
for name, dest in [("phase1_ROPEFIX_295M_weights.pt", "best.pt"), ("phase1_tokenizer.json", "tokenizer.json"),
                   ("phase1_run_config.json", "run_config.json")]:
    if not os.path.exists(f"{W}/{dest}"):
        print("downloading", name, flush=True)
        urllib.request.urlretrieve(REL + name, f"{W}/{dest}")
sys.path[:0] = [LLM, f"{CAP}/shared"]
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| bf16:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## Main recipe (frozen model + sentence model)

In [ ]:
sh(f"cd {CAP}/Phase3_RR/src && python embed.py --backbone {W} --out ../runs/emb")
sh(f"cd {CAP}/Phase3_RR/src && python train_rr.py --emb ../runs/emb --out_dir ../runs/rr_nb" + (" --epochs 2" if QUICK else ""))

## LoRMA run: adapters only in the top 8 of 23 blocks (5 held-out folds)

In [ ]:
folds = [0] if QUICK else [0, 1, 2, 3, 4]
for k in folds:
    sh(f"cd {CAP}/Phase3_RR/src && python finetune.py --backbone {W} --fold {k} --context 256 --max_sent 256 --lr 5e-4 "
       f"--lorma_rank 16 --lorma_alpha 32 --lorma_top 8 --no_extract --out ../runs/lorma/top8_k{k}"
       + (" --epochs 1" if QUICK else ""))

## Scores

In [ ]:
for f in sorted(glob.glob(f"{CAP}/Phase3_RR/runs/**/*.json", recursive=True))[-20:]:
    print(f)